In [1]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

In [2]:
import sys
sys.path.append("..") 

In [3]:
from processing import load_UCI_dataset,extract_all_features,extract_features_from_dataset, select_f_test, select_mrmr, select_reliefF, save_feature_set

In [4]:
X_train, y_train, X_val, y_val, X_test, y_test = load_UCI_dataset(WINDOW_SIZE= 1000, STEP_SIZE= 1000)

Total recordings: 12000
Train recordings: 9600
Validation recordings: 1200
Test recordings: 1200


100%|██████████| 9600/9600 [01:08<00:00, 140.72it/s]


Skipped recordings: 4


100%|██████████| 1200/1200 [00:08<00:00, 136.94it/s]


Skipped recordings: 1


100%|██████████| 1200/1200 [00:08<00:00, 139.01it/s]

Skipped recordings: 0


In [5]:
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (266201, 1, 1000), y_train shape: (266201, 2)


## Feature Exraction - Non-fiducial features ##

We'll extract 19 features

In [6]:
test_window = X_train[0, 0, :]

features = extract_all_features(
    test_window,
    fs=125
)

print("Number of features:", len(features))

print(features)

Number of features: 57
{'ppg_mean': np.float64(2.136435972629521), 'ppg_median': np.float64(2.1940371456500487), 'ppg_std': np.float64(0.5766428331561362), 'ppg_variance': np.float64(0.33251695703033557), 'ppg_iqr': np.float64(1.0349462365591395), 'ppg_skewness': np.float64(-0.20699780736329657), 'ppg_kurtosis': np.float64(-1.223775673674285), 'ppg_zero_crossing_rate': np.float64(0.0), 'ppg_shannon_entropy': np.float64(5.535366320192241), 'ppg_energy_mean': np.float64(4.896875622175783), 'ppg_energy_variance': np.float64(5.817547072959762), 'ppg_energy_skewness': np.float64(0.08741581435573596), 'ppg_energy_kurtosis': np.float64(-1.319225971606975), 'ppg_energy_iqr': np.float64(4.440504515402821), 'ppg_kte_mean': np.float64(0.005852562856363285), 'ppg_kte_variance': np.float64(0.00022669556843995535), 'ppg_kte_skewness': np.float64(0.12138201774612678), 'ppg_kte_kurtosis': np.float64(0.8579770784977327), 'ppg_kte_iqr': np.float64(0.014694528293052789), 'vpg_mean': np.float64(-0.0002153

In [7]:
df_train = extract_features_from_dataset(
    X_train,
    y_train
)

df_val = extract_features_from_dataset(
    X_val,
    y_val
)

df_test = extract_features_from_dataset(
    X_test,
    y_test
)

  0%|          | 0/266201 [00:00<?, ?it/s]

100%|██████████| 33746/33746 [05:37<00:00, 99.91it/s] 


In [8]:
print("Train:", df_train.shape)
print("Validation:", df_val.shape)
print("Test:", df_test.shape)

Train: (266201, 59)
Validation: (33605, 59)
Test: (33746, 59)


## Feature Selection ##

We'll reduce the 57 features to 15 using 3 different methods

In [9]:
feature_columns = [
    col for col in df_train.columns
    if col not in ["SBP", "DBP"]
]

X_train = df_train[feature_columns]
X_val = df_val[feature_columns]
X_test = df_test[feature_columns]

y_train_sbp = df_train["SBP"]
y_train_dbp = df_train["DBP"]

y_val_sbp = df_val["SBP"]
y_val_dbp = df_val["DBP"]

y_test_sbp = df_test["SBP"]
y_test_dbp = df_test["DBP"]

## F-Test ##

In [10]:
# SBP
f_sbp_features, f_sbp_results = select_f_test(
    X_train,
    y_train_sbp,
    k=15
)
print("F-test selected features for SBP:")

for i, feature in enumerate(f_sbp_features, 1):
    print(i, feature)

F-test selected features for SBP:
1 vpg_skewness
2 apg_zero_crossing_rate
3 apg_skewness
4 ppg_iqr
5 apg_shannon_entropy
6 vpg_shannon_entropy
7 ppg_energy_iqr
8 vpg_energy_skewness
9 ppg_variance
10 vpg_kurtosis
11 vpg_median
12 ppg_energy_variance
13 ppg_std
14 vpg_iqr
15 ppg_kurtosis


In [11]:
# DBP
f_dbp_features, f_dbp_results = select_f_test(
    X_train,
    y_train_dbp,
    k=15
)

print("\nF-test selected features for DBP:")

for i, feature in enumerate(f_dbp_features, 1):
    print(i, feature)


F-test selected features for DBP:
1 apg_iqr
2 vpg_median
3 ppg_kte_iqr
4 apg_kte_iqr
5 apg_kte_mean
6 vpg_kte_mean
7 apg_variance
8 apg_energy_mean
9 apg_std
10 apg_energy_iqr
11 vpg_kte_iqr
12 apg_median
13 ppg_kte_mean
14 ppg_skewness
15 vpg_variance


## mRMR ##

In [12]:
mrmr_sbp_features, mrmr_sbp_results = select_mrmr(
    X_train,
    y_train_sbp,
    k=15
)

print("mRMR selected features for SBP:")

for i, feature in enumerate(
    mrmr_sbp_features,
    1
):
    print(i, feature)

mRMR: using 50000 samples for feature selection
mRMR selected features for SBP:
1 apg_energy_variance
2 ppg_zero_crossing_rate
3 ppg_energy_skewness
4 vpg_skewness
5 ppg_energy_variance
6 apg_mean
7 apg_energy_skewness
8 apg_zero_crossing_rate
9 vpg_mean
10 apg_median
11 vpg_median
12 apg_skewness
13 ppg_kte_kurtosis
14 apg_kte_variance
15 ppg_shannon_entropy


In [13]:
mrmr_dbp_features, mrmr_dbp_results = select_mrmr(
    X_train,
    y_train_dbp,
    k=15
)

print("\nmRMR selected features for DBP:")

for i, feature in enumerate(
    mrmr_dbp_features,
    1
):
    print(i, feature)

mRMR: using 50000 samples for feature selection

mRMR selected features for DBP:
1 apg_energy_variance
2 ppg_zero_crossing_rate
3 ppg_energy_skewness
4 ppg_energy_variance
5 apg_zero_crossing_rate
6 apg_mean
7 ppg_kte_kurtosis
8 vpg_mean
9 apg_median
10 apg_kte_variance
11 vpg_skewness
12 ppg_shannon_entropy
13 apg_skewness
14 vpg_kte_kurtosis
15 vpg_median


## ReliefF ##

In [14]:
relief_sbp_features, relief_sbp_results = select_reliefF(
    X_train,
    y_train_sbp,
    k=15
)

print("ReliefF selected features for SBP:")

for i, feature in enumerate(
    relief_sbp_features,
    1
):
    print(i, feature)

RReliefF: using 50000 samples for feature selection


ReliefF selected features for SBP:
1 vpg_mean
2 apg_mean
3 apg_zero_crossing_rate
4 apg_median
5 ppg_shannon_entropy
6 apg_skewness
7 vpg_median
8 vpg_skewness
9 vpg_shannon_entropy
10 vpg_zero_crossing_rate
11 ppg_skewness
12 ppg_kurtosis
13 ppg_energy_kurtosis
14 apg_shannon_entropy
15 ppg_energy_skewness


In [15]:
relief_dbp_features, relief_dbp_results = select_reliefF(
    X_train,
    y_train_dbp,
    k=15
)

print("\nReliefF selected features for DBP:")

for i, feature in enumerate(
    relief_dbp_features,
    1
):
    print(i, feature)

RReliefF: using 50000 samples for feature selection

ReliefF selected features for DBP:
1 vpg_mean
2 apg_mean
3 apg_zero_crossing_rate
4 apg_median
5 ppg_shannon_entropy
6 apg_skewness
7 vpg_skewness
8 vpg_median
9 vpg_shannon_entropy
10 ppg_skewness
11 vpg_zero_crossing_rate
12 ppg_kurtosis
13 ppg_energy_kurtosis
14 apg_shannon_entropy
15 vpg_iqr


In [16]:
selection_summary = {

    "SBP_F_test": f_sbp_features,

    "SBP_mRMR": mrmr_sbp_features,

    "SBP_ReliefF": relief_sbp_features,

    "DBP_F_test": f_dbp_features,

    "DBP_mRMR": mrmr_dbp_features,

    "DBP_ReliefF": relief_dbp_features
}

In [17]:
for name, features in selection_summary.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    for i, feature in enumerate(features, 1):
        print(f"{i:2d}. {feature}")


SBP_F_test
 1. vpg_skewness
 2. apg_zero_crossing_rate
 3. apg_skewness
 4. ppg_iqr
 5. apg_shannon_entropy
 6. vpg_shannon_entropy
 7. ppg_energy_iqr
 8. vpg_energy_skewness
 9. ppg_variance
10. vpg_kurtosis
11. vpg_median
12. ppg_energy_variance
13. ppg_std
14. vpg_iqr
15. ppg_kurtosis

SBP_mRMR
 1. apg_energy_variance
 2. ppg_zero_crossing_rate
 3. ppg_energy_skewness
 4. vpg_skewness
 5. ppg_energy_variance
 6. apg_mean
 7. apg_energy_skewness
 8. apg_zero_crossing_rate
 9. vpg_mean
10. apg_median
11. vpg_median
12. apg_skewness
13. ppg_kte_kurtosis
14. apg_kte_variance
15. ppg_shannon_entropy

SBP_ReliefF
 1. vpg_mean
 2. apg_mean
 3. apg_zero_crossing_rate
 4. apg_median
 5. ppg_shannon_entropy
 6. apg_skewness
 7. vpg_median
 8. vpg_skewness
 9. vpg_shannon_entropy
10. vpg_zero_crossing_rate
11. ppg_skewness
12. ppg_kurtosis
13. ppg_energy_kurtosis
14. apg_shannon_entropy
15. ppg_energy_skewness

DBP_F_test
 1. apg_iqr
 2. vpg_median
 3. ppg_kte_iqr
 4. apg_kte_iqr
 5. apg_kte_

## Saving ##

In [18]:
save_feature_set(
    name="SBP_FTEST",
    selected_features=f_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="SBP_MRMR",
    selected_features=mrmr_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="SBP_RELIEFF",
    selected_features=relief_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

Processing: SBP_FTEST
Number of selected features: 15


Train shape: (266201, 16)
Val shape:   (33605, 16)
Test shape:  (33746, 16)

Saved:
data/SBP_FTEST_train.csv
data/SBP_FTEST_val.csv
data/SBP_FTEST_test.csv
data/SBP_FTEST_preprocessing.pkl

Processing: SBP_MRMR
Number of selected features: 15
Train shape: (266201, 16)
Val shape:   (33605, 16)
Test shape:  (33746, 16)

Saved:
data/SBP_MRMR_train.csv
data/SBP_MRMR_val.csv
data/SBP_MRMR_test.csv
data/SBP_MRMR_preprocessing.pkl

Processing: SBP_RELIEFF
Number of selected features: 15
Train shape: (266201, 16)
Val shape:   (33605, 16)
Test shape:  (33746, 16)

Saved:
data/SBP_RELIEFF_train.csv
data/SBP_RELIEFF_val.csv
data/SBP_RELIEFF_test.csv
data/SBP_RELIEFF_preprocessing.pkl



In [19]:
save_feature_set(
    name="DBP_FTEST",
    selected_features=f_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="DBP_MRMR",
    selected_features=mrmr_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="DBP_RELIEFF",
    selected_features=relief_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

Processing: DBP_FTEST
Number of selected features: 15
Train shape: (266201, 16)
Val shape:   (33605, 16)
Test shape:  (33746, 16)

Saved:
data/DBP_FTEST_train.csv
data/DBP_FTEST_val.csv
data/DBP_FTEST_test.csv
data/DBP_FTEST_preprocessing.pkl

Processing: DBP_MRMR
Number of selected features: 15
Train shape: (266201, 16)
Val shape:   (33605, 16)
Test shape:  (33746, 16)

Saved:
data/DBP_MRMR_train.csv
data/DBP_MRMR_val.csv
data/DBP_MRMR_test.csv
data/DBP_MRMR_preprocessing.pkl

Processing: DBP_RELIEFF
Number of selected features: 15
Train shape: (266201, 16)
Val shape:   (33605, 16)
Test shape:  (33746, 16)

Saved:
data/DBP_RELIEFF_train.csv
data/DBP_RELIEFF_val.csv
data/DBP_RELIEFF_test.csv
data/DBP_RELIEFF_preprocessing.pkl

